<a href="https://colab.research.google.com/github/DiyaMalik07/SC_Project/blob/main/MLRan_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip install -q pymoo

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.core.callback import Callback
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.operators.crossover.hux import HalfUniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize

required_files = ["MLRan_X_train_RFE.csv", "MLRan_X_test_RFE.csv"]
for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing {f}! Please upload it to the Colab files panel.")

print("Loading and combining datasets...")
train_df = pd.read_csv("MLRan_X_train_RFE.csv", low_memory=False)
test_df = pd.read_csv("MLRan_X_test_RFE.csv", low_memory=False)

# Combine them (they already contain the labels)
df = pd.concat([train_df, test_df], ignore_index=True)

if 'type_label' in df.columns:
    y = (df['type_label'] > 0).astype(int).values
elif 'family_label' in df.columns:
    y = (df['family_label'] > 0).astype(int).values
else:
    # Fallback just in case
    y = (df.iloc[:, -1] > 0).astype(int).values

cols_to_drop = ['sample_id', 'sample_type', 'family_label', 'type_label', 'label']
X_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
X = X_df.apply(pd.to_numeric, errors='coerce').fillna(0).values

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
POPULATION_SIZE = 40
GENERATIONS = 30
n_features = X.shape[1]

print(f"Class balance -> Ransomware: {sum(y==1)} | Goodware: {sum(y==0)}")

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp)

print("Training Baseline Random Forest...")
start_time = time.time()
baseline_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)
baseline_train_time = time.time() - start_time

baseline_val_f1 = f1_score(y_val, baseline_model.predict(X_val), zero_division=0)
baseline_accuracy = accuracy_score(y_test, baseline_model.predict(X_test))
baseline_f1 = f1_score(y_test, baseline_model.predict(X_test), zero_division=0)

class DiverseBinarySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        pop = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            pop[i] = np.random.rand(problem.n_var) < np.random.uniform(0.02, 0.80)
            if pop[i].sum() == 0: pop[i, np.random.randint(0, problem.n_var)] = True
        return pop

class SingleObjProblem(Problem):
    def __init__(self, alpha=0.95):
        super().__init__(n_var=n_features, n_obj=1, xl=0, xu=1, vtype=bool)
        self.alpha = alpha
    def _evaluate(self, pop, out, *args, **kwargs):
        fitness = []
        for ind in pop:
            mask = ind.astype(bool)
            if mask.sum() == 0: mask[np.random.randint(0, n_features)] = True
            model = RandomForestClassifier(n_estimators=30, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train[:, mask], y_train)
            f1 = f1_score(y_val, model.predict(X_val[:, mask]), zero_division=0)
            fitness.append((self.alpha * (1 - f1)) + ((1 - self.alpha) * (mask.sum() / n_features)))
        out["F"] = np.column_stack([fitness])

def evaluate_subset(mask, name, run_time):
    t0 = time.time()
    model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train[:, mask], y_train)
    preds = model.predict(X_test[:, mask])
    return {"Algorithm": name, "Accuracy": accuracy_score(y_test, preds), "F1": f1_score(y_test, preds, zero_division=0), "Features": int(mask.sum()), "Reduction %": (1 - mask.sum() / n_features) * 100, "Search Time": run_time, "Train Time": time.time() - t0}

results_log = []
print("Setup complete.")

Loading and combining datasets...
Class balance -> Ransomware: 2330 | Goodware: 2550
Training Baseline Random Forest...
Setup complete.


In [34]:
print("Running SPEA2...")
class SPEA2Problem(Problem):
    def __init__(self):
        super().__init__(n_var=n_features, n_obj=2, xl=0, xu=1, vtype=bool)
    def _evaluate(self, pop, out, *args, **kwargs):
        f1_loss, frac = [], []
        for ind in pop:
            mask = ind.astype(bool)
            if mask.sum() == 0: mask[np.random.randint(0, n_features)] = True
            model = RandomForestClassifier(n_estimators=30, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train[:, mask], y_train)
            f1_loss.append(1 - f1_score(y_val, model.predict(X_val[:, mask]), zero_division=0))
            frac.append(mask.sum() / n_features)
        out["F"] = np.column_stack([f1_loss, frac])

t0 = time.time()
res_spea2 = minimize(SPEA2Problem(), SPEA2(pop_size=POPULATION_SIZE, sampling=DiverseBinarySampling(), crossover=HalfUniformCrossover(), mutation=BitflipMutation(prob=5.0/n_features), eliminate_duplicates=True), termination=('n_gen', GENERATIONS), seed=RANDOM_SEED)
spea2_time = time.time() - t0

best_idx = np.argmin(res_spea2.F[:, 0])
spea2_mask = res_spea2.X[best_idx].astype(bool)

results_log.append(evaluate_subset(spea2_mask, "SPEA2", spea2_time))
print(f"SPEA2 finished. Selected {spea2_mask.sum()} features.")

Running SPEA2...
SPEA2 finished. Selected 88 features.


In [35]:
print("Running SGA...")
t0 = time.time()
res_sga = minimize(SingleObjProblem(), GA(pop_size=POPULATION_SIZE, sampling=DiverseBinarySampling(), crossover=HalfUniformCrossover(), mutation=BitflipMutation(prob=5.0/n_features), eliminate_duplicates=True), termination=('n_gen', GENERATIONS), seed=RANDOM_SEED)
results_log.append(evaluate_subset(res_sga.X.astype(bool), "SGA", time.time() - t0))
print("SGA finished.")

Running SGA...
SGA finished.


In [36]:
print("Running AGA...")
class AGACallback(Callback):
    def notify(self, algo):
        F = algo.pop.get("F")
        if F is not None and len(F) > 0:
            algo.mating.mutation = BitflipMutation(prob=min(0.5, (5.0/n_features)*5) if (np.mean(F) - np.min(F)) < 0.01 else 5.0/n_features)

t0 = time.time()
res_aga = minimize(SingleObjProblem(), GA(pop_size=POPULATION_SIZE, sampling=DiverseBinarySampling(), crossover=HalfUniformCrossover(prob=0.9), mutation=BitflipMutation(prob=5.0/n_features), eliminate_duplicates=True), termination=('n_gen', GENERATIONS), callback=AGACallback(), seed=RANDOM_SEED)
results_log.append(evaluate_subset(res_aga.X.astype(bool), "AGA", time.time() - t0))
print("AGA finished.")

Running AGA...
AGA finished.


In [37]:
print("Running Memetic GA...")
class MemeticCallback(Callback):
    def notify(self, algo):
        pop, prob = algo.pop, algo.problem
        F_vals = pop.get("F")
        if F_vals is None or len(F_vals) == 0: return
        best_idx = np.argmin(F_vals)
        best_X = pop[best_idx].get("X").astype(bool)
        neighbors = []
        for _ in range(3):
            n = best_X.copy()
            flip = np.random.randint(0, prob.n_var)
            n[flip] = not n[flip]
            if n.sum() == 0: n[np.random.randint(0, prob.n_var)] = True
            neighbors.append(n)
        out = {}
        prob._evaluate(np.array(neighbors), out)
        best_n_idx = np.argmin(out["F"])
        if out["F"][best_n_idx][0] < F_vals[best_idx]:
            pop[best_idx].set("X", neighbors[best_n_idx])
            pop[best_idx].set("F", out["F"][best_n_idx])

t0 = time.time()
res_mga = minimize(SingleObjProblem(), GA(pop_size=POPULATION_SIZE, sampling=DiverseBinarySampling(), crossover=HalfUniformCrossover(), mutation=BitflipMutation(prob=5.0/n_features), eliminate_duplicates=True), termination=('n_gen', GENERATIONS), callback=MemeticCallback(), seed=RANDOM_SEED)
results_log.append(evaluate_subset(res_mga.X.astype(bool), "Memetic GA", time.time() - t0))
print("Memetic GA finished.")

Running Memetic GA...
Memetic GA finished.


In [38]:
baseline_dict = {"Algorithm": "Baseline", "Accuracy": baseline_accuracy, "F1": baseline_f1, "Features": n_features, "Reduction %": 0.0, "Search Time": 0.0, "Train Time": baseline_train_time}
print("\n" + "="*80)
print("FINAL RESULTS: MLRan DATASET")
print("="*80)
print(pd.DataFrame([baseline_dict] + results_log).to_string(index=False))


FINAL RESULTS: MLRan DATASET
 Algorithm  Accuracy       F1  Features  Reduction %  Search Time  Train Time
  Baseline  0.989754 0.989293       483     0.000000     0.000000    0.387398
     SPEA2  0.990779 0.990395        88    81.780538   127.070570    0.291956
       SGA  0.984631 0.983854       103    78.674948   147.246683    0.315905
       AGA  0.985656 0.985043        84    82.608696   140.337965    0.304245
Memetic GA  0.984631 0.983854       107    77.846791   145.182568    0.337151
